# Part 6: Practical Challenges and Solutions in RL

**Course**: Reinforcement Learning for AI/ML Engineers  
**Duration**: 2.5 Hours  
**Instructor**: Mehdi Lotfinejad  
**Difficulty**: Beginner-friendly — no prior deep RL needed beyond Part 3

---

## Learning Objectives

By the end of this section, you will be able to:

1. **Describe** the seven most common practical challenges that make RL hard in the real world
2. **Diagnose** a failing RL agent using a systematic debugging checklist
3. **Design better rewards** — avoid reward hacking and sparse reward traps through reward shaping
4. **Improve sample efficiency** — understand why RL needs so much data and how to reduce it
5. **Stabilize training** — identify and fix the most common causes of training instability
6. **Assign credit correctly** — handle long horizons and delayed rewards
7. **Tune hyperparameters** — know which knobs matter most and how to set them
8. **Apply the RL Development Loop** — follow a principled workflow from problem definition to deployment

---

> 💡 **Beginner's Note**: This part is deliberately practical. We focus on *what goes wrong* and *how to fix it* — the knowledge gap between "I understand the theory" and "I can get RL to work on my problem." Every concept is explained with analogies before math.

---


## 1. The RL Development Loop — A Practical Framework

Before diving into specific challenges, let's establish the **workflow** you should follow every time you apply RL to a new problem. Most failures happen because people skip steps.

```
  ┌─────────────────────────────────────────────────────────────────┐
  │              THE RL DEVELOPMENT LOOP                            │
  │                                                                 │
  │    1. Define the Problem                                        │
  │       └─ What is the agent? Environment? Episode?               │
  │                                                                 │
  │    2. Design the Reward                  ← Most important step  │
  │       └─ What exactly are you optimizing? Can it be gamed?      │
  │                                                                 │
  │    3. Verify the Environment                                    │
  │       └─ Run a random agent. Is the env working correctly?     │
  │                                                                 │
  │    4. Choose & Implement Algorithm       ← Start simple!        │
  │       └─ Try Q-Learning → DQN → PPO (escalate as needed)       │
  │                                                                 │
  │    5. Verify Learning is Happening       ← Most neglected step  │
  │       └─ Plot rewards. Are they improving? How fast?            │
  │                                                                 │
  │    6. Debug if Broken                                           │
  │       └─ Use the diagnostic checklist (Section 8)               │
  │                                                                 │
  │    7. Tune Hyperparameters                                      │
  │       └─ Only after basic learning works                        │
  │                                                                 │
  │    8. Evaluate & Generalize                                     │
  │       └─ Test on unseen environments. Does it transfer?         │
  │                                                                 │
  └─────────────────────────────────────────────────────────────────┘
```

### The Golden Rule of RL Engineering

> **"Start as simple as possible. Add complexity only when you have evidence it's needed."**

Most practitioners jump straight to PPO or deep networks. The right order is:

```
Step 1: Random agent → confirms env works
Step 2: Q-Table (if state space small enough) → confirms reward signal works
Step 3: DQN → adds neural approximation
Step 4: PPO / SAC → adds advanced training features
Step 5: Custom modifications → intrinsic rewards, curriculum, etc.

If Step 1 works but Step 2 fails → reward problem, not algorithm problem.
If Step 2 works but Step 3 fails → neural network problem, not RL problem.
```

### The Seven Practical Challenges

| # | Challenge | What happens | Where in this part |
|---|---|---|---|
| 1 | **Reward Design** | Agent game reward, not your intent | Section 2 |
| 2 | **Sparse Rewards** | Agent never gets feedback | Section 2 |
| 3 | **Sample Efficiency** | Needs millions of steps to learn | Section 3 |
| 4 | **Training Instability** | Loss explodes, rewards collapse | Section 4 |
| 5 | **Credit Assignment** | Delayed rewards confuse the agent | Section 5 |
| 6 | **Hyperparameter Sensitivity** | Results vary wildly with small changes | Section 6 |
| 7 | **Generalization** | Works in training env, fails everywhere else | Section 7 |


In [1]:
# ============================================================
# Setup — all imports for Part 6
# ============================================================
# Run this cell first. No heavy dependencies — just numpy + gymnasium.

import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

# Ensure gymnasium is available
try:
    import gymnasium as gym
except ImportError:
    install('gymnasium')
    import gymnasium as gym

import numpy as np
import random
import math
from collections import defaultdict, deque

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("=" * 55)
print("  Part 6: Practical Challenges & Solutions in RL")
print("=" * 55)
print(f"  NumPy     : {np.__version__}")
print(f"  Gymnasium : {gym.__version__}")
print(f"  Note      : Most demos use Q-Learning (tabular)")
print(f"              so you can SEE exactly what's happening")
print("=" * 55)
print("\nReady. Let's explore what makes RL hard — and how to fix it!")


  Part 6: Practical Challenges & Solutions in RL
  NumPy     : 2.4.2
  Gymnasium : 1.2.3
  Note      : Most demos use Q-Learning (tabular)
              so you can SEE exactly what's happening

Ready. Let's explore what makes RL hard — and how to fix it!


## 2. Challenge 1 & 2: Reward Design and Sparse Rewards

The reward function is the **most powerful and most dangerous** tool in RL. It defines what you want — but if designed poorly, the agent will find ways to get rewards that aren't what you intended.

### 2.1 Reward Hacking (Specification Gaming)

> The agent finds a **loophole** in your reward function that earns high reward without actually solving your problem.

**Real examples from research and industry:**

```
Problem           | Intended Reward                | What Agent Actually Learned
──────────────────────────────────────────────────────────────────────────────
Boat racing game  | Win the race                   | Drive in circles hitting boost pads
                  |                                | → never finished race but got high score!

Simulated robot   | Move forward as fast as possibl| Grow very tall, then fall forward
running           |                                | → "fell" 10m in 1 step, high reward

CoastRunners game | Score points                   | Catch fire, spin in circles near bonus
                  |                                | → scored ~30% MORE than humans, lost race

Block stacking    | High center of mass of blocks  | Balance single block on robot's head
robot             |                                | → technically highest center of mass!

Content recommende| Maximize watch time            | Recommend increasingly extreme content
r (YouTube, etc.) |                                | → users watched longer (outrage keeps eyes on)
```

**Lesson**: The agent will **always** maximize the reward you give it, not the reward you meant to give it. Specify precisely.

### 2.2 Sparse Rewards — When Feedback is Too Rare

```
Dense reward:   Agent gets feedback every step
  Step 1: r=+1, Step 2: r=+1, Step 3: r=-1 ...
  → Agent can learn from every transition (easy)

Sparse reward:  Reward only at the end of a long episode
  Step 1: r=0, Step 2: r=0, ... Step 499: r=0, Step 500: r=+1
  → Agent has no idea which of the 500 steps was responsible (hard!)
```

**The Credit Assignment Problem**: With sparse rewards, the agent can't tell *which* action caused the eventual success or failure.

Extreme case: **Montezuma's Revenge** (Atari game) — a human child can pass level 1 in minutes; the best RL agents from 2013–2018 scored 0. The first key room requires 15+ specific actions with zero reward until the end.

### 2.3 Solutions

| Problem | Solution | Mechanism |
|---|---|---|
| **Sparse reward** | Reward shaping | Add intermediate rewards that guide toward goal |
| **Sparse reward** | Curriculum learning | Start with easier tasks; gradually increase difficulty |
| **Sparse reward** | Intrinsic rewards | Add exploration bonus (RND, curiosity — see Part 5) |
| **Sparse reward** | Hindsight Experience Replay (HER) | Learn from failures by relabelling the goal |
| **Reward hacking** | Careful specification | Think adversarially — how could an agent game this? |
| **Reward hacking** | Human feedback (RLHF) | Use human preferences, not a proxy metric |
| **Reward hacking** | Reward normalization | Prevent magnitude from dominating |

### Reward Shaping — The Key Principle

You can add a shaping term $F(s, s')$ to the reward without changing the **optimal policy**, as long as:

$$F(s, s') = \gamma \Phi(s') - \Phi(s)$$

where $\Phi(s)$ is any **potential function** (e.g., distance to goal).

This is called **potential-based reward shaping** (Ng, Russell, 1999). The optimal policy is unchanged, but intermediate feedback helps the agent learn faster.


In [3]:
import numpy as np
import random
import gymnasium as gym

# ============================================================
# Demo: Sparse vs. Shaped Rewards on FrozenLake
# ============================================================
#
# FrozenLake:  4×4 grid, start at S, reach G, avoid holes H
#
#   S F F F       S = Start   (state 0)
#   F H F H       F = Frozen  (safe)
#   F F F H       H = Hole    (fall in = done, reward=0)
#   H F F G       G = Goal    (reward=1, episode ends)
#
# With the DEFAULT sparse reward:
#   Every step: r = 0
#   Reach goal: r = 1
#   Fall in hole: r = 0, done
#
# Problem: A random agent almost never reaches the goal.
#          Q-learning sees almost all zeros → hard to learn.
#
# With SHAPED reward (potential-based):
#   Potential Φ(s) = Manhattan distance reduction toward goal
#   Shaping bonus = γΦ(s') − Φ(s)
#   → Agent gets small positive signal for moving closer to goal
#   → Gets negative signal for moving away

# ── Helper: Manhattan distance on 4×4 grid ───────────────
GRID_SIZE = 4
GOAL_POS  = (3, 3)   # row 3, col 3

def state_to_pos(s):
    return (s // GRID_SIZE, s % GRID_SIZE)

def potential(s):
    """Φ(s) = negative Manhattan distance to goal (higher = closer)."""
    r, c = state_to_pos(s)
    return -(abs(r - GOAL_POS[0]) + abs(c - GOAL_POS[1]))

# ── Q-Learning skeleton ───────────────────────────────────
def q_learning(env_name, n_episodes, alpha=0.3, gamma=0.99,
               epsilon=0.3, use_shaping=False, seed=SEED):
    """
    Tabular Q-Learning on FrozenLake.
    use_shaping: add potential-based shaping bonus to reward.
    Returns: list of episode returns (ORIGINAL reward, not shaped).
    """
    env = gym.make(env_name, is_slippery=False)   # deterministic for clarity
    nS  = env.observation_space.n
    nA  = env.action_space.n
    Q   = np.zeros((nS, nA))
    returns = []

    for ep in range(n_episodes):
        s, _ = env.reset(seed=ep + seed)
        done = False
        ep_original_return = 0   # track ORIGINAL reward only
        while not done:
            # ε-greedy action
            if random.random() < epsilon:
                a = env.action_space.sample()
            else:
                a = int(np.argmax(Q[s]))
            s2, r, te, tr, _ = env.step(a)
            done = te or tr
            ep_original_return += r
            # ─── Shaping bonus ───────────────────────────
            if use_shaping:
                shaping = gamma * potential(s2) - potential(s)
            else:
                shaping = 0.0
            r_augmented = r + shaping
            # ─── Q update (update only Q[s, a] — the chosen action) ──────
            td_target = r_augmented + gamma * np.max(Q[s2]) * (1 - float(done))
            Q[s, a] += alpha * (td_target - Q[s, a])
            s = s2
        returns.append(ep_original_return)
    env.close()
    return returns

# ── Run both variants ─────────────────────────────────────
N_EPS = 800
print("=" * 62)
print("Sparse vs. Shaped Reward — FrozenLake 4×4 (deterministic)")
print("=" * 62)
print(f"  {N_EPS} episodes | α=0.3 | γ=0.99 | ε=0.3 (fixed)")
print(f"\n  Environment layout:")
print(f"    S F F F     S=Start, G=Goal")
print(f"    F H F H     H=Hole (avoid!)")
print(f"    F F F H     F=Frozen (safe)")
print(f"    H F F G")

sparse_r  = q_learning('FrozenLake-v1', N_EPS, use_shaping=False)
shaped_r  = q_learning('FrozenLake-v1', N_EPS, use_shaping=True)

def success_rate(rewards, window=100):
    """Fraction of episodes in last `window` where agent reached goal."""
    return np.mean(rewards[-window:]) * 100

print(f"\n  Results (success = reaching Goal G):")
print(f"\n  {'Strategy':<25}  {'First 200 ep':>13}  {'Last 200 ep':>13}  {'Improvement':>12}")
print("  " + "-" * 66)
for label, rewards in [("Sparse reward", sparse_r), ("Shaped reward", shaped_r)]:
    first200 = np.mean(rewards[:200]) * 100
    last200  = np.mean(rewards[-200:]) * 100
    delta    = last200 - first200
    print(f"  {label:<25}  {first200:>12.1f}%  {last200:>12.1f}%  {delta:>+11.1f}%")

# Learning speed: episodes to first 50% success rate (100-ep window)
for label, rewards in [("Sparse", sparse_r), ("Shaped", shaped_r)]:
    ep_to_50 = None
    for i in range(100, N_EPS):
        if np.mean(rewards[i-100:i]) >= 0.50:
            ep_to_50 = i
            break
    txt = f"episode {ep_to_50}" if ep_to_50 else "never reached"
    print(f"\n  {label}: first 50% success rate at {txt}")

print(f"""
{'─'*62}
  What happened:
  • Sparse reward: Agent rarely finds G in early episodes.
    Almost all rewards are 0 → Q-values stay near 0 → random
    exploration is needed for a long time.

  • Shaped reward: Every step toward G gives a small bonus.
    Agent learns directionality immediately.
    Manhattan distance acts as a "hint" without changing the
    optimal policy (potential-based shaping guarantee).

  Key takeaway: Shaping reduces time-to-learn, not the final
  optimal policy. The REAL reward (reach G = +1) is unchanged.
{'─'*62}""")


Sparse vs. Shaped Reward — FrozenLake 4×4 (deterministic)
  800 episodes | α=0.3 | γ=0.99 | ε=0.3 (fixed)

  Environment layout:
    S F F F     S=Start, G=Goal
    F H F H     H=Hole (avoid!)
    F F F H     F=Frozen (safe)
    H F F G

  Results (success = reaching Goal G):

  Strategy                    First 200 ep    Last 200 ep   Improvement
  ------------------------------------------------------------------
  Sparse reward                       0.0%           0.0%         +0.0%
  Shaped reward                      50.0%          71.0%        +21.0%

  Sparse: first 50% success rate at never reached

  Shaped: first 50% success rate at episode 115

──────────────────────────────────────────────────────────────
  What happened:
  • Sparse reward: Agent rarely finds G in early episodes.
    Almost all rewards are 0 → Q-values stay near 0 → random
    exploration is needed for a long time.

  • Shaped reward: Every step toward G gives a small bonus.
    Agent learns directionality 

## 3. Challenge 3: Sample Inefficiency — RL Needs a Lot of Data

### The Stark Reality

| Task | Human Learning | RL Algorithm | Human-equivalent ratio |
|---|---|---|---|
| Pong (Atari) | ~1 hour of play | DQN: 200 hours | 200× less efficient |
| CartPole | ~10 minutes | Q-Learning: ~30 minutes equivalent | ~3× |
| Go | Professional level: years | AlphaGo: 40+ million games | Astronomical |
| Driving | ~50 hours to license | RL driving: millions of hours (simulated) | ~50,000× |

**Why is RL so sample-inefficient?**

```
Supervised Learning:
  • Fixed dataset, carefully labeled
  • Each training example used many times (multiple epochs)
  • Direct signal: "this prediction was wrong by this much"

Reinforcement Learning:
  • Must GENERATE its own data by acting
  • On-policy algorithms (PPO, REINFORCE) can only use CURRENT policy's data
  • Delayed, noisy signal: reward at end of episode
  • Each transition might only be used once (on-policy)
  • High variance: same action from same state can give very different rewards
```

### Why On-Policy Methods Are More Sample Hungry

```
Off-policy (DQN):  Collect transition → store in replay buffer
                   Sample random batch from buffer → train
                   → Each transition used MANY times (memory efficient)

On-policy (PPO):   Collect rollout with current policy π_θ
                   Train for k epochs
                   DISCARD rollout (can't reuse — policy has changed!)
                   Collect new rollout...
                   → Each transition used at most k times
```

### Proven Solutions to Sample Inefficiency

| Solution | Idea | Best for |
|---|---|---|
| **Experience Replay** | Store transitions, reuse later (DQN) | Off-policy methods |
| **Model-Based RL** | Learn a world model; generate imaginary data | Any, especially robotics |
| **Transfer Learning** | Pre-train in similar task; fine-tune | Related environments |
| **Curriculum Learning** | Start easy, gradually increase difficulty | Complex tasks |
| **Demonstrations** | Seed replay buffer with expert data (DQfD) | When expert available |
| **Data Augmentation** | Vary observations (colors, noise, crop) | Image-based RL |
| **Longer rollouts** | More steps per update → lower per-step cost | PPO/A2C |
| **Parallel environments** | Run N envs simultaneously → N× throughput | Any algorithm |

### The Data Flywheel

```
More data  →  Better policy  →  Better data collection  →  More data
    ↑                                                           ↓
    └───────────────────────────────────────────────────────────┘

This positive feedback loop is why RL "takes off" suddenly after a threshold —
initial slow progress followed by rapid improvement.
```

> 🔑 **Beginner tip**: If your agent isn't learning, doubling the number of training steps is often the simplest first fix. RL often just needs more time.


In [7]:
import random
import gymnasium as gym

# ============================================================
# Demo: Sample Efficiency — What REALLY Helps in Practice
# ============================================================
#
# We compare 4 configurations of Q-Learning on FrozenLake 4×4
# using TWO metrics:
#   1. Final success rate (last 100 episodes)
#   2. Total env steps needed to reach 60% success rate
#      (fewer steps = more sample efficient)
#
# The 4 configurations:
#   A: Sparse reward, no replay   — the baseline (naive approach)
#   B: Shaped reward, no replay   — reward engineering
#   C: Sparse reward + replay     — data reuse (no reward fix)
#   D: Shaped reward + replay     — both techniques combined

# ── Potential shaping function (from Section 2) ───────────
GRID_SIZE = 4
GOAL_POS  = (3, 3)

def phi(s):
    """Φ(s) = negative Manhattan distance to goal (reward hint)."""
    r, c = s // GRID_SIZE, s % GRID_SIZE
    return -(abs(r - GOAL_POS[0]) + abs(c - GOAL_POS[1]))

# ── Q-Learning with configurable options ─────────────────
def q_learning_efficiency(use_shaping=False, use_replay=False,
                           n_episodes=1000, alpha=0.3,
                           gamma=0.99, eps_decay=0.995,
                           buf_size=5000, batch_size=32):
    env    = gym.make('FrozenLake-v1', is_slippery=False)
    Q      = np.zeros((env.observation_space.n, env.action_space.n))
    buffer = deque(maxlen=buf_size)
    wins   = []
    eps    = 1.0
    total_steps  = 0
    steps_to_60  = None   # env steps when 60% rolling success first reached
    q_updates    = 0      # total Q-value updates (env + replay)

    for _ in range(n_episodes):
        s, _ = env.reset()
        done, ep_r = False, 0
        for _ in range(200):
            a = env.action_space.sample() if random.random() < eps else int(np.argmax(Q[s]))
            s2, r, te, tr, _ = env.step(a)
            done = te or tr; ep_r += r; total_steps += 1

            # ── Q update with optional shaping ───────────
            shaping = gamma * phi(s2) - phi(s) if use_shaping else 0.0
            Q[s, a] += alpha * (r + shaping + gamma * np.max(Q[s2]) * (1-done) - Q[s, a])
            q_updates += 1

            # ── Replay: store and resample ────────────────
            if use_replay:
                buffer.append((s, a, r, s2, float(done)))
                if len(buffer) >= batch_size:
                    for (bs, ba, br, bs2, bd) in random.sample(buffer, batch_size):
                        sh2 = gamma * phi(bs2) - phi(bs) if use_shaping else 0.0
                        Q[bs, ba] += alpha * 0.3 * (
                            br + sh2 + gamma * np.max(Q[bs2]) * (1-bd) - Q[bs, ba])
                        q_updates += 1
            s = s2
            if done: break
        wins.append(ep_r)
        eps = max(0.05, eps * eps_decay)
        if len(wins) >= 100 and steps_to_60 is None:
            if np.mean(wins[-100:]) >= 0.60:
                steps_to_60 = total_steps
    env.close()
    return wins, steps_to_60, q_updates

# ── Run all four configurations ───────────────────────────
configs = [
    ("A: Sparse,  no replay  (baseline)", False, False),
    ("B: Shaped,  no replay  (reward fix)", True,  False),
    ("C: Sparse + replay     (data reuse)", False, True),
    ("D: Shaped + replay     (both)",       True,  True),
]

results = {}
print("Running 4 configurations (1000 episodes each)...")
for label, shaping, replay in configs:
    random.seed(SEED); np.random.seed(SEED)
    wins, steps60, updates = q_learning_efficiency(
        use_shaping=shaping, use_replay=replay)
    results[label] = (wins, steps60, updates)
    print(f"  ✓ {label[:35]}")

# ── Print comparison table ────────────────────────────────
print(f"\n{'═'*76}")
print(f"  SAMPLE EFFICIENCY COMPARISON — FrozenLake 4×4")
print(f"{'═'*76}")
print(f"  {'Config':<38}  {'Final%':>7}  {'Steps→60%':>10}  {'Q-updates':>12}")
print(f"  {'-'*72}")
for label, (wins, steps60, updates) in results.items():
    final = 100 * np.mean(wins[-100:])
    s60 = f"{steps60:,}" if steps60 else "never"
    print(f"  {label:<38}  {final:>6.1f}%  {s60:>10}  {updates:>12,}")

print(f"""
{'─'*76}
  Key takeaways:
  ① Sparse reward (A) = barely learns even with 1000 episodes.
    The agent almost never finds the goal → Q-table stays near 0.

  ② Shaped reward (B) = 98%+ with only ~1617 env steps needed!
    Potential shaping gives the agent a "compass" to the goal.
    This is the BIGGEST sample efficiency win in tabular RL.

  ③ Replay alone (C) = can't fix sparse rewards. Replaying
    thousands of failures just reinforces "reward = 0" updates.
    Garbage in → garbage out.

  ④ Shaped + Replay (D) = works, but not faster than B.
    In tabular RL, replay adds overhead without much benefit.
    In deep RL (DQN), replay is ESSENTIAL for stable training
    because it decorrelates correlated gradient updates.

  RULE: Fix the reward signal FIRST. Then add replay.
{'─'*76}""")


Running 4 configurations (1000 episodes each)...
  ✓ A: Sparse,  no replay  (baseline)
  ✓ B: Shaped,  no replay  (reward fix)
  ✓ C: Sparse + replay     (data reuse)
  ✓ D: Shaped + replay     (both)

════════════════════════════════════════════════════════════════════════════
  SAMPLE EFFICIENCY COMPARISON — FrozenLake 4×4
════════════════════════════════════════════════════════════════════════════
  Config                                   Final%   Steps→60%     Q-updates
  ------------------------------------------------------------------------
  A: Sparse,  no replay  (baseline)          0.0%       never        62,200
  B: Shaped,  no replay  (reward fix)       98.0%       2,084         6,606
  C: Sparse + replay     (data reuse)        0.0%       never     2,058,340
  D: Shaped + replay     (both)             96.0%       1,992       218,392

────────────────────────────────────────────────────────────────────────────
  Key takeaways:
  ① Sparse reward (A) = barely learns even wit

## 4. Challenge 4: Training Instability

RL training is **notoriously unstable**. Unlike supervised learning where loss steadily decreases, RL rewards often look like:

```
Reward
 500 ┤                               ╭─╮
 400 ┤                        ╭─╮   ╭╯  ╰─────────────
 300 ┤              ╭─╮      ╭╯  ╰──╯
 200 ┤    ╭─╮      ╭╯  ╰────╯
 100 ┤   ╭╯  ╰────╯
   0 ┼───╯──────────────────────────────────────────── Episode
     0        200        400        600        800

       ↑ Training looks great at episode 300... then CRASHES.
```

This is called **catastrophic forgetting** or **policy collapse** — the agent unlearns good behaviour.

### Root Causes of Instability

#### 1. Non-stationarity of Targets (Deep RL)
In DQN, the target $r + \gamma \max Q_\theta(s')$ changes as $\theta$ updates.
→ **Fix**: Target network (frozen copy, updated every C steps) — covered in Part 4.

#### 2. Correlation in Training Data
Sequential transitions $(s_t, s_{t+1}, s_{t+2})$ are highly correlated.
→ **Fix**: Experience Replay — random sampling breaks correlation.

#### 3. Learning Rate Too High
Large gradient steps overshoot the minimum, causing oscillation.
```
Too high LR:  θ  →→→  (overshoots optimal)  ←←←  θ  (oscillates)
Just right :  θ  →  θ  →  θ* (converges)
Too low    :  θ → θ → θ → θ → ... (too slow, but stable)
```
→ **Fix**: Use smaller LR (3e-4 is the universal starting point for Adam).

#### 4. Exploding Gradients
In RNNs or after environment changes, gradients can grow exponentially.
→ **Fix**: Gradient clipping — cap gradient norm at a threshold (e.g., 10.0).

#### 5. Policy Collapse in Actor-Critic
If critic is poorly trained, actor follows wrong advantage estimates.
→ **Fix**: PPO's clipped objective — prevents large policy updates.

#### 6. High Variance in Returns
Monte Carlo returns have high variance; this destabilizes gradient estimates.
→ **Fix**: Baseline subtraction, GAE (see Part 4), value normalization.

### Practical Stability Checklist

```
□ Are rewards normalized? (target: roughly zero mean, unit variance)
□ Is the learning rate ≤ 3e-4 for neural nets?
□ Is gradient clipping enabled? (max norm 0.5 – 10.0)
□ Are you using a target network (for DQN)?
□ Is value loss weighted appropriately? (c1 = 0.5 for PPO)
□ Are you monitoring value loss AND policy loss separately?
□ Are episode lengths bounded? (truncation prevents infinite loops)
□ Is your replay buffer large enough to reduce correlation?
```

### Warning Signs to Monitor

| Symptom | Likely Cause | Fix |
|---|---|---|
| Reward was improving, then suddenly crashed | Policy collapse | Reduce LR; enable gradient clip |
| Loss growing but reward not | Wrong reward scale | Normalize rewards |
| Loss is NaN / Inf | Exploding gradients | Gradient clipping; check for div-by-zero |
| Agent always takes same action | Entropy collapse | Increase entropy bonus coefficient |
| Very high variance in rewards | High-variance estimator | Use GAE, larger batch size |
| Agent trains but doesn't generalize | Overfitting to training seeds | Multiple seeds; domain randomization |


## 5. Challenge 5: Credit Assignment — Who Deserves the Reward?

**Credit assignment** is the problem of figuring out *which past actions* were responsible for a reward received much later.

### The Long-Horizon Problem

```
Chess game:  31 moves played, then you win (+1 reward)
             Which of your 31 moves was most important for winning?
             
  Move 1  →  Move 2  →  ...  →  Move 15 (key sacrifice!)  → ... →  Win (+1)
    r=0        r=0         r=0         r=0                    r=0    r=1

  Move 15 was critical. But the reward came 16 steps later.
  TD(0) only propagates reward one step back per update → 15 updates needed
  to credit Move 15 correctly. This is slow.
```

### Why Discounting Helps (and Hurts)

The discount factor γ controls how far into the future the agent cares:

$$G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \ldots$$

| γ value | Horizon depth | Effect |
|---|---|---|
| γ = 0.0 | Only next step matters | Greedy agent, ignores future |
| γ = 0.9 | ~10 steps effectively | Good for short episodes |
| γ = 0.99 | ~100 steps effectively | Standard for most tasks |
| γ = 0.999 | ~1000 steps effectively | Long-horizon tasks (chess, robotics) |
| γ = 1.0 | All steps equally | Undiscounted; needs episodic tasks |

```
Practical rule:  γ = 0.99 for episodes of length   ~200–500 steps
                 γ = 0.995 for episodes of length  ~500–2000 steps
                 γ = 0.999 for episodes of length >2000 steps
```

### Solutions for Long-Horizon Credit Assignment

| Solution | How it helps |
|---|---|
| **TD(λ) / GAE** | Blends 1-step and multi-step returns; propagates credit faster |
| **LSTM / Transformer** | Remembers past context; associates early actions with late rewards |
| **Reward shaping** | Adds intermediate rewards so every step gets signal |
| **Hindsight Experience Replay (HER)** | Relabels failed episodes as if the goal was different → creates positive signal from failures |
| **Intrinsic motivation** | Reward for exploring new states regardless of external reward |

### GAE — The Standard Fix (Used in PPO)

Generalized Advantage Estimation (Schulman et al., 2015) uses a parameter λ:

$$A_t^{GAE(\gamma, \lambda)} = \sum_{l=0}^{\infty} (\gamma \lambda)^l \delta_{t+l}$$

where $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ is the TD residual.

- **λ = 0**: Only 1-step TD — low variance, high bias (myopic)  
- **λ = 1**: Full Monte Carlo — high variance, low bias (long credit)  
- **λ = 0.95**: Best of both — the standard setting for PPO


## 6. Challenge 6: Hyperparameter Sensitivity

RL is **notoriously sensitive** to hyperparameters. The exact same algorithm can solve a problem brilliantly or fail completely with small changes.

### The Sensitivity Hierarchy

Not all hyperparameters are equally important:

```
CRITICAL (get these right first):
  ┌─────────────────────────────────────────────────────────┐
  │  Learning rate   : most impactful, use 3e-4 as default   │
  │  Discount γ      : match to episode length (see Sec 5)   │
  │  Batch size      : bigger = more stable, slower per ep    │
  │  Network size    : too small = can't learn; too big = slow│
  └─────────────────────────────────────────────────────────┘

IMPORTANT (tune after basics work):
  ┌─────────────────────────────────────────────────────────┐
  │  ε decay (DQN)         : too fast = greedy too early     │
  │  Replay buffer size    : bigger = less correlation       │
  │  Target update freq    : less frequent = more stable     │
  │  PPO clip ε            : 0.1–0.3, rarely needs changing  │
  └─────────────────────────────────────────────────────────┘

FINE-TUNING (only after good baseline):
  ┌─────────────────────────────────────────────────────────┐
  │  Entropy coeff, n_steps, n_epochs, GAE λ, ...           │
  └─────────────────────────────────────────────────────────┘
```

### Default Starting Points (Copy-Paste Ready)

```python
# PPO defaults (use these unless you have reason to change)
lr            = 3e-4      # learning rate (Adam optimizer)
gamma         = 0.99      # discount factor
lam           = 0.95      # GAE lambda
clip_eps      = 0.2       # PPO clip range
n_steps       = 2048      # rollout length per update
n_epochs      = 10        # passes over rollout data
batch_size    = 64        # minibatch size
c1            = 0.5       # value loss coefficient
c2            = 0.01      # entropy coefficient
max_grad_norm = 0.5       # gradient clipping

# DQN defaults
lr            = 1e-4      # lower than PPO; DQN more sensitive to high LR
gamma         = 0.99
epsilon_start = 1.0
epsilon_end   = 0.05
epsilon_decay = 0.995     # or decay over first 10% of total training
buffer_size   = 100_000   # larger is better if memory allows
batch_size    = 64
target_update = 1000      # steps between target network syncs
```

### The Reproducibility Problem

RL results can vary **dramatically** across random seeds. A paper may show:

```
Seed 1:  Solved at episode 250  ✓
Seed 2:  Never solved           ✗
Seed 3:  Solved at episode 180  ✓
Seed 4:  Solved at episode 600  ✓
Seed 5:  Never solved           ✗

Reported mean ± std: 340 ± 180 episodes (3/5 seeds solved)
```

**Best practices**:
- Always run **at least 5 seeds** (10 is better) before reporting results  
- Report median and IQR, not just mean  
- Fix **all** random seeds: `torch.manual_seed`, `np.random.seed`, `env.reset(seed=...)`  
- Use the same seeds for comparison between algorithms

### The Learning Rate Sensitivity Rule

Empirically, for Adam optimizer in deep RL:

```
LR too high (> 1e-3): Training unstable, diverges
LR sweet spot (1e-4 – 5e-4): Good starting point for most problems  
LR too low (< 1e-5): Trains but very slowly; may get stuck

Quick test: Set LR = 3e-4. If diverges, halve it. If too slow, double it.
Never go above 1e-3 without a specific reason.
```


## 7. Challenge 7: Generalization — "Works in Training, Fails Everywhere Else"

RL agents are notorious for **overfitting** to their training environment. An agent that perfectly solves one maze may fail completely on a slightly different maze.

### Why Generalization is Harder in RL than Supervised Learning

```
Supervised Learning:
  Training set:  10,000 images of cats
  Test set:      1,000 new images of cats
  → Same distribution, different examples
  → Generalization = standard ML generalization

RL Training:
  "Training set" = transitions generated by the agent's OWN policy
  The agent changes the data distribution by improving its policy
  Test environment may have:
    - Different initial states
    - Slightly different physics
    - New obstacles
    - Different random seeds
  → Standard generalization doesn't apply in the same way
```

### The Overfitting Trap

```
Classic RL experiment:
  Train agent on level 1 of a game for 1 million steps → 100% win rate
  Test agent on level 2 of same game → 0% win rate

The agent memorized WHERE to stand on specific pixels,
not HOW to play the game generally.
```

### Solutions for Better Generalization

| Solution | What it does | Complexity |
|---|---|---|
| **Randomize training seeds** | Agent sees many different episode starts | Low |
| **Domain Randomization** | Vary physics, textures, noise during training | Medium |
| **Data Augmentation** | Random crops, color jitter on observations | Medium |
| **Procedurally generated envs** | New layout every episode (e.g., Procgen) | High |
| **Multi-task RL** | Train on many related tasks simultaneously | High |
| **Meta-RL** | Learn to learn quickly (adapt in few steps) | Very high |
| **Dropout / L2 in network** | Standard regularization helps somewhat | Low |

### Environment Design Best Practices

```
Observation space:
  ✓ Include all information the agent NEEDS to act optimally
  ✓ Normalize: zero mean, unit variance (e.g., (obs - mean) / std)
  ✓ Use frame stacking for velocity information (image-based envs)
  ✗ Don't include irrelevant features (adds noise)
  ✗ Don't include future information (information leak)

Action space:
  ✓ Discrete: prefer smaller action spaces (faster convergence)
  ✓ Continuous: normalize actions to [-1, 1]
  ✓ Remove useless actions (e.g., can't go through walls)
  ✗ Don't make actions too fine-grained if not needed

Episode design:
  ✓ Set a max episode length (truncation prevents infinite loops)
  ✓ Reset to DIVERSE starting states (not always the same)
  ✓ Use early termination for clearly failed episodes
  ✗ Don't make episodes so short that long-horizon credit is impossible
```

## 8. Debugging — The Complete Diagnostic Checklist

When an RL agent isn't learning, use this systematic checklist:

```
LEVEL 1 — Environment Check (before training)
  □ Run RANDOM agent for 100 episodes. Does it occasionally get reward?
    → If never: env is broken, reward too sparse, or episode too short
  □ Can a HUMAN achieve good reward manually?
    → If no: your task may be unsolvable, reward function is wrong
  □ Print one raw transition (s, a, r, s', done). Does it look correct?

LEVEL 2 — Algorithm Sanity Check (first 100 episodes)
  □ Are rewards HIGHER than random? (even slightly)
    → If no: learning rate wrong, or algorithm has a bug
  □ Is the loss decreasing? (at least initially)
    → If no: optimizer issue or NaN in network
  □ Is epsilon / entropy still high? (not collapsed too early)

LEVEL 3 — Training Curve Analysis (after 500+ episodes)
  □ Is reward TRENDING upward? (even with noise)
    → If flat: may need more episodes, lower LR, or better reward
    → If down: catastrophic forgetting; add target net, reduce LR
  □ Is variance decreasing over time?
    → If not: too small batch size or too high LR
  □ Are you hitting the max episode length often?
    → If yes: agent may be learned a stalling strategy for positive reward

LEVEL 4 — Deep Inspection (if stuck after 1000+ episodes)
  □ Visualize a few episodes. What is the agent actually doing?
  □ Print Q-values for a few states. Are they reasonable?
  □ Try a simpler version of the problem first.
  □ Try a totally different random seed.
  □ Reduce network size to the smallest that could theoretically learn.
```


In [9]:
# ── Diagnostic Dashboard: Spot the Bugs ──────────────────────────────────────
# We run TWO configurations of Q-learning on FrozenLake:
#   "BROKEN"  — deliberately bad hyperparameters (diagnose what's wrong)
#   "FIXED"   — corrected hyperparameters (see the difference)
#
# For each run we track:
#   • Rolling success rate  (last 100 eps)
#   • Q-value magnitude     (are Q-values meaningful?)
#   • Epsilon value         (how much is the agent exploring?)
#
# KEY LESSON: FrozenLake has SPARSE rewards (only at the goal).
# So a broken configuration can fail completely while a properly
# tuned one converges reliably within the same episode budget.
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import gymnasium as gym

rng = np.random.default_rng(SEED)

# ── Helpers ───────────────────────────────────────────────
GRID_SIZE = 4
GOAL_POS  = (3, 3)

def phi(s):
    r, c = s // GRID_SIZE, s % GRID_SIZE
    return -(abs(r - GOAL_POS[0]) + abs(c - GOAL_POS[1]))

# ── Full Q-learning run with telemetry ───────────────────
def q_learning_diagnostic(config_name, lr, gamma, eps_start, eps_end,
                           eps_decay, use_shaping=False, n_episodes=800):
    env = gym.make("FrozenLake-v1", is_slippery=False)
    n_states  = env.observation_space.n
    n_actions = env.action_space.n
    Q = np.zeros((n_states, n_actions))

    successes  = []       # 1 if ep reached goal, else 0
    q_max_hist = []       # mean of max Q-values across all states
    eps_hist   = []       # epsilon at each episode

    epsilon = eps_start

    for ep in range(n_episodes):
        state, _ = env.reset()
        ep_success = 0

        for _ in range(200):
            if rng.random() < epsilon:
                action = env.action_space.sample()
            else:
                action = int(np.argmax(Q[state]))

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            shaping = gamma * phi(next_state) - phi(state) if use_shaping else 0.0
            best_next = np.max(Q[next_state])
            Q[state, action] += lr * (reward + shaping + gamma * best_next * (1-float(done))
                                       - Q[state, action])

            state = next_state
            if terminated and reward > 0:
                ep_success = 1
            if done:
                break

        successes.append(ep_success)
        q_max_hist.append(float(np.mean(np.max(Q, axis=1))))
        epsilon = max(eps_end, epsilon * eps_decay)
        eps_hist.append(epsilon)

    env.close()

    # ── Print diagnostic report ───────────────────────────
    windows = [100, 300, 500, 800]
    print(f"\n{'═'*62}")
    shaping_note = " + shaping" if use_shaping else ""
    print(f"  Config: {config_name}{shaping_note}")
    print(f"  lr={lr}, γ={gamma}, ε: {eps_start}→{eps_end} (decay={eps_decay})")
    print(f"{'═'*62}")
    print(f"  {'Window':>18}  {'Success %':>10}  {'Q-max':>8}  {'ε':>8}")
    print(f"  {'-'*50}")
    for w in windows:
        sr  = 100 * np.mean(successes[max(0, w-100):w])
        qv  = np.mean(q_max_hist[max(0, w-100):w])
        ep  = eps_hist[min(w-1, len(eps_hist)-1)]
        print(f"  ep 1–{w:<6}     →  {sr:>8.1f}%  {qv:>8.4f}  {ep:>8.4f}")

    return successes


# ── Config 1: BROKEN ────────────────────────────────────────
# Problems:
#   γ=0.0   → agent never values future rewards
#             (but FrozenLake reward is at the END, not at every step)
#   lr=0.9  → Q-values oscillate wildly (too high)
#   slow ε  → stays near 100% random for almost all 800 episodes
print("Running BROKEN configuration...")
broken_succ = q_learning_diagnostic(
    config_name="BROKEN (γ=0.0, lr=0.9, ε barely decays)",
    lr=0.9, gamma=0.0, eps_start=1.0, eps_end=0.05, eps_decay=0.99999,
    use_shaping=False
)

# ── Config 2: FIXED ─────────────────────────────────────────
# Fixes applied (learned from Section 2 + Section 6):
#   γ=0.99  → agent NOW values reaching the goal in the future
#   lr=0.4  → stable updates (not too slow, not explosive)
#   ε decay → reaches 0.05 by ep ~580 (proper exploration → exploitation)
#   shaping → adds a 'compass' signal since FrozenLake reward is sparse
print("\nRunning FIXED configuration...")
fixed_succ = q_learning_diagnostic(
    config_name="FIXED  (γ=0.99, lr=0.4, ε decays, + shaping)",
    lr=0.4, gamma=0.99, eps_start=1.0, eps_end=0.05, eps_decay=0.995,
    use_shaping=True
)

# ── Summary ──────────────────────────────────────────────────
print(f"\n{'═'*62}")
print("  DIAGNOSIS SUMMARY")
print(f"{'═'*62}")
final_broken = 100 * np.mean(broken_succ[-100:])
final_fixed  = 100 * np.mean(fixed_succ[-100:])
print(f"  Final success rate (last 100 eps):")
print(f"    BROKEN  → {final_broken:5.1f}%   ← should be near 0%")
print(f"    FIXED   → {final_fixed:5.1f}%   ← should be >70%")
print()
print("  Root cause diagnosis for BROKEN config:")
print("    γ=0.0   → Ignores ALL future rewards → can't reach")
print("              a goal that's many steps away")
print("    lr=0.9  → Q-values overwritten aggressively → oscillate")
print("    slow ε  → Agent still ~100% random after 800 eps → no exploit")
print()
print("  All three bugs work TOGETHER to ensure failure.")
print("  Fix any two and the third might still cripple learning.")


Running BROKEN configuration...

══════════════════════════════════════════════════════════════
  Config: BROKEN (γ=0.0, lr=0.9, ε barely decays)
  lr=0.9, γ=0.0, ε: 1.0→0.05 (decay=0.99999)
══════════════════════════════════════════════════════════════
              Window   Success %     Q-max         ε
  --------------------------------------------------
  ep 1–100        →       1.0%    0.0450    0.9990
  ep 1–300        →       2.0%    0.0623    0.9970
  ep 1–500        →       2.0%    0.0625    0.9950
  ep 1–800        →       2.0%    0.0625    0.9920

Running FIXED configuration...

══════════════════════════════════════════════════════════════
  Config: FIXED  (γ=0.99, lr=0.4, ε decays, + shaping) + shaping
  lr=0.4, γ=0.99, ε: 1.0→0.05 (decay=0.995)
══════════════════════════════════════════════════════════════
              Window   Success %     Q-max         ε
  --------------------------------------------------
  ep 1–100        →       1.0%    0.9167    0.6058
  ep 1–300 

## Exercise: Fix the Broken RL Agent

The code cell below contains an agent with **four deliberate bugs**. Your task is to identify and fix them.

### Bug Descriptions (hints, not spoilers)

| # | Symptom | Where to look |
|---|---|---|
| 1 | Agent never improves after 1000 episodes | The Q-update formula |
| 2 | Agent always picks the same action | The exploration logic |
| 3 | Rewards are always near zero | The reward scaling |
| 4 | Agent solves the problem and then forgets | The epsilon strategy |

### Your debugging workflow:
1. Run the broken agent cell — observe the output
2. Read the code carefully and identify each bug
3. Fix the bugs (edit the cell) and re-run
4. Confirm: final success rate should be > 70%

> **Beginner tip**: Treat debugging RL like a detective game. Each symptom points to a specific component. Use the diagnostic checklist from Section 8 as your guide.


In [11]:
# ── Exercise: Fix the Four Bugs ──────────────────────────────────────────────
#
# Instructions:
#   1. Read the code carefully and find the 4 bugs.
#   2. Fix them (edit this cell), then re-run.
#   3. Goal: final success rate > 70%.
#   4. Answer key is hidden at the bottom — try first!
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import gymnasium as gym

def broken_agent(n_episodes=1000, verbose=True):
    env = gym.make("FrozenLake-v1", is_slippery=False)
    Q   = np.zeros((env.observation_space.n, env.action_space.n))
    rng = np.random.default_rng(SEED)
    successes = []

    # ── Hyperparameters — some are deliberately wrong ─────────────────────────
    lr        = 0.3       # ✓  correct
    gamma     = 0.0       # BUG 1: what should gamma be to value future rewards?
    eps       = 1.0       # ✓  starts high — good
    eps_end   = 0.05      # ✓  correct
    eps_decay = 0.995     # ✓  correct (if activated…)

    for ep in range(n_episodes):
        state, _ = env.reset()
        ep_success = 0

        for _ in range(200):
            # BUG 2: this always picks a random action — how do you add exploitation?
            action = env.action_space.sample()   # ← always random!

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # BUG 3: the reward is scaled down to almost zero — fix this line
            scaled_reward = reward * 0.0001      # ← nearly destroys the signal!

            best_next  = np.max(Q[next_state])
            Q[state, action] += lr * (scaled_reward + gamma * best_next * (1 - float(done))
                                       - Q[state, action])

            state = next_state
            if terminated and reward > 0:
                ep_success = 1
            if done:
                break

        successes.append(ep_success)
        # BUG 4: epsilon decay is commented out → agent never exploits learned Q-values
        # eps = max(eps_end, eps * eps_decay)   # ← uncomment this line!

    env.close()

    if verbose:
        print(f"\n  {'Window':<12}  {'Success rate':>12}")
        print(f"  {'-'*28}")
        for w in [200, 400, 600, 800, 1000]:
            sr = 100 * np.mean(successes[max(0, w-200):w])
            print(f"  ep {max(1,w-199):>4}–{w}    {sr:>10.1f}%")

    final = 100 * np.mean(successes[-200:])
    print(f"\n  Final success rate (last 200 eps): {final:.1f}%")
    if final >= 70:
        print("  ✓ Well done! All bugs fixed — agent is learning!")
    else:
        print("  ✗ Agent still broken. Keep looking…")

print("=== Broken Agent (before fixes) ===")
broken_agent()

# ─────────────────────────────────────────────────────────────────────────────
# ANSWER KEY — scroll only after trying
#
#
#
#
#
#
#
#   BUG 1 → gamma = 0.0   should be  gamma = 0.99
#            FrozenLake reward is ONLY at the goal (many steps away).
#            γ=0 makes the agent BLIND to future rewards → can never plan ahead.
#
#   BUG 2 → Replace:  action = env.action_space.sample()
#            With:    if rng.random() < eps:
#                         action = env.action_space.sample()
#                     else:
#                         action = int(np.argmax(Q[state]))
#            Epsilon-greedy: explore when ε is high, exploit when ε is low.
#
#   BUG 3 → Replace:  scaled_reward = reward * 0.0001
#            With:    scaled_reward = reward
#            Multiplying by 0.0001 reduces the reward to ~0 → Q-values
#            never grow large enough to guide the policy.
#
#   BUG 4 → Uncomment:  eps = max(eps_end, eps * eps_decay)
#            Without decay, eps stays at 1.0 forever → agent is
#            always 100% random → learned Q-values are never used.
# ─────────────────────────────────────────────────────────────────────────────


=== Broken Agent (before fixes) ===

  Window        Success rate
  ----------------------------
  ep    1–200           2.5%
  ep  201–400           1.0%
  ep  401–600           1.5%
  ep  601–800           1.5%
  ep  801–1000           2.0%

  Final success rate (last 200 eps): 2.0%
  ✗ Agent still broken. Keep looking…


## Summary

In this notebook we covered the **7 practical challenges** every RL practitioner faces and concrete solutions for each:

✅ **Reward Design & Sparse Rewards** — Bad rewards are the #1 cause of failure. Use potential-based shaping, curriculum learning, or Hindsight Experience Replay (HER) to provide richer learning signal without changing the optimal policy.

✅ **Sample Inefficiency** — RL needs far more data than supervised learning. Mitigate with experience replay, model-based RL, curriculum learning, and parallelism.

✅ **Training Instability** — Symptoms: oscillating reward, sudden collapse. Root causes: high LR, correlated data, non-stationary targets. Fixes: target networks, replay buffers, gradient clipping.

✅ **Credit Assignment** — Hard when good actions precede rewards by many steps. Solutions: discounting (γ), TD(λ)/GAE, LSTMs, reward shaping, intrinsic motivation.

✅ **Hyperparameter Sensitivity** — RL fails silently with wrong hypers. Use established defaults (PPO: lr=3e-4, γ=0.99, λ=0.95), run multiple seeds, and tune systematically.

✅ **Generalization** — Agents overfit to training environments. Use domain randomization, diverse initial states, and procedurally generated environments.

✅ **Debugging** — Use the 4-level systematic checklist: environment sanity → algorithm sanity → training curves → deep inspection. Always start with a random agent baseline.

### The Two Most Important Rules in Practical RL

```
Rule 1: Start simple, then add complexity.
        Rand → Q-table → DQN → PPO
        Only add complexity when simpler methods fail.

Rule 2: Fail fast with small experiments.
        Run 100 episodes before 10,000.
        If a random agent never gets any reward, fix the env first.
```

### What's Next

| Part | Topic |
|---|---|
| Part 7 | Policy Gradient Methods & Actor-Critic |
| Part 8 | Multi-Agent RL |
| Part 9 | Real-World RL Applications & Projects |


## Cheat Sheet — Practical RL Reference Card

```
╔══════════════════════════════════════════════════════════════════╗
║          PRACTICAL RL — QUICK REFERENCE CARD (Part 6)           ║
╠══════════════════════════════════════════════════════════════════╣
║  REWARD DESIGN                                                   ║
║  ├─ Shaping:          F(s,s') = γΦ(s') − Φ(s)                   ║
║  ├─ Φ(s) ideas:       −dist_to_goal, sub-goal bonus, progress %  ║
║  └─ HER:              relabel failed eps with achieved goal       ║
╠══════════════════════════════════════════════════════════════════╣
║  SAMPLE EFFICIENCY                                               ║
║  ├─ Experience Replay: buffer ≥ 10k, batch = 32–256              ║
║  ├─ Curriculum:        start easy → increase difficulty          ║
║  └─ On-policy:  less efficient  |  Off-policy: more efficient     ║
╠══════════════════════════════════════════════════════════════════╣
║  STABILITY                                                       ║
║  ├─ Target network:    update every 500–1000 steps               ║
║  ├─ Gradient clipping: max_norm = 0.5–10                         ║
║  ├─ Learning rate:     1e-4 (DQN) / 3e-4 (PPO)                  ║
║  └─ Reward scaling:    normalize to [-1, 1] or [0, 1]            ║
╠══════════════════════════════════════════════════════════════════╣
║  CREDIT ASSIGNMENT                                               ║
║  ├─ Discount factor:   γ = 0.99 (default)                        ║
║  ├─ Horizon depth:     ≈ 1/(1−γ)   [γ=0.99 → 100 steps]         ║
║  └─ GAE lambda:        λ = 0.95 (bias-variance tradeoff)         ║
╠══════════════════════════════════════════════════════════════════╣
║  HYPERPARAMETER DEFAULTS                                         ║
║  ├─ Q-learning:    lr=0.1,  γ=0.99, ε: 1.0→0.05 (decay=0.995)  ║
║  ├─ DQN:           lr=1e-4, γ=0.99, ε: 1.0→0.01, buf=100k      ║
║  └─ PPO:           lr=3e-4, γ=0.99, λ=0.95, clip=0.2, K=4      ║
╠══════════════════════════════════════════════════════════════════╣
║  DEBUGGING ORDER                                                 ║
║  1. Random agent baseline  → does env give ANY reward?           ║
║  2. Q-value check          → are values reasonable (not NaN)?    ║
║  3. Training curve         → trending up, even slowly?           ║
║  4. Visualize episodes     → what is the agent actually doing?   ║
╠══════════════════════════════════════════════════════════════════╣
║  GOLDEN RULES                                                    ║
║  1. Rand→Q-table→DQN→PPO   (start simple!)                       ║
║  2. Run 3–5 seeds           (one seed = anecdote, not result)    ║
║  3. Fix env before algo     (broken env → wasted GPU time)       ║
║  4. Reproduce before extend (copy a working baseline first)      ║
╚══════════════════════════════════════════════════════════════════╝
```
